In [17]:
from dotenv import load_dotenv
load_dotenv()

True

In [29]:
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

from pydantic import BaseModel, Field
from langgraph.graph import START, END, StateGraph

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [ ]:
# load doc:
docs = PyPDFLoader('../data/SDE_Resume_Sandip.pdf').load()


# Split docs in chunks:
split_docs = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100).split_documents(docs)

# vector embbedings:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

# store in vector DB:
vector_store = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    persist_directory="./agentic_rag_store"
)

In [20]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
)

In [21]:
# structured output:
class RAGState(BaseModel):
    question: str
    documents: list = []
    context: str = Field(default="")
    answer: str = Field(default="")

In [22]:
# user query -> retrive from vector db -> get context -> generate o/p -> end

def retrive_node(state: RAGState)->RAGState:
    docs = vector_store.similarity_search(query=state.question)
    state.documents = docs
    
    return state

def context_node(state:RAGState)-> RAGState:
    context =""
    
    for doc in state.documents:
        context += doc.page_content + "\n\n"
        
    state.context = context
    
    return state

def generate_node(state:RAGState) -> RAGState:
    prompt = f"""
        You are a assistant and provide the answer for user question based on the provided con
        If you don't find the relevant answer, then just say 'I dont know.'.
        Context is: {state.context},
        Questions is: {state.question}
    """
    res = llm.invoke(prompt)
    state.answer = res.content
    
    return state

In [23]:
graph = StateGraph(RAGState)

graph.add_node("retrive_node", retrive_node)
graph.add_node("context_node", context_node)
graph.add_node("generate_node", generate_node)

In [24]:
graph.add_edge(START,  "retrive_node")
graph.add_edge("retrive_node", "context_node")
graph.add_edge("context_node", "generate_node")
graph.add_edge("generate_node", END)

graph = graph.compile()

In [33]:
res = graph.invoke({"question":"is he currently working?"})
res["answer"]

'No, he is not currently working.'